# Negotiation Environment Demo
Strategic Language in Agentic Markets — Single episode + condition comparison

In [ ]:
# Install dependencies
!pip install openai -q

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # ← your key here

In [ ]:
# If running from repo root:
import sys
sys.path.insert(0, '.')

from negotiation_project import (
    Condition, NegotiationConfig, NegotiationEnv,
    sample_types, summarize,
    run_batch, run_condition_comparison, save_results
)

## 1. Single episode (Free-form)

In [ ]:
config = NegotiationConfig(
    condition=Condition.FREE_FORM,
    max_rounds=8,
    lambda_selfishness=0.5,
)

buyer_type, seller_type = sample_types(seed=42)
print(f"Buyer: {buyer_type}")
print(f"Seller: {seller_type}")
print(f"Surplus zone: ${seller_type.reservation_price:.0f} – ${buyer_type.reservation_price:.0f}")

In [ ]:
env = NegotiationEnv(config)
result = env.run(buyer_type=buyer_type, seller_type=seller_type)

print(f"\nOutcome: {'DEAL' if result.deal_reached else 'NO DEAL'} ({result.termination})")
if result.deal_reached:
    print(f"Price: ${result.deal_price:.0f} | Round: {result.deal_round}")
    print(f"Buyer surplus: {result.buyer_surplus:.1f}")
    print(f"Seller surplus: {result.seller_surplus:.1f}")
    print(f"Total welfare: {result.total_welfare:.1f}")

In [ ]:
# Print full transcript
print("\n--- TRANSCRIPT ---")
for turn in result.turns:
    print(f"\n[Round {turn.round} | {turn.speaker.upper()} | action={turn.action} price={turn.price}]")
    print(f"Visible: {turn.visible_message}")

## 2. Condition comparison (RQ2 core)

In [ ]:
base_config = NegotiationConfig(
    max_rounds=8,
    lambda_selfishness=0.5,
)

# Run 5 episodes per condition with same random seeds (same private types)
all_results = run_condition_comparison(
    n_episodes=5,
    base_config=base_config,
    conditions=list(Condition),
    base_seed=0,
    verbose=True,
)

In [ ]:
import pandas as pd

rows = []
for cond, results in all_results.items():
    s = summarize(results)
    rows.append({
        "condition": cond,
        "agreement_rate": f"{s['agreement_rate']:.0%}",
        "avg_deal_round": f"{s['avg_deal_round']:.1f}",
        "avg_buyer_surplus": f"{s['avg_buyer_surplus']:.1f}",
        "avg_seller_surplus": f"{s['avg_seller_surplus']:.1f}",
        "avg_welfare_all_eps": f"{s['avg_total_welfare_all']:.1f}",
    })

pd.DataFrame(rows).set_index("condition")

In [ ]:
# Save full results to JSON
from negotiation_project.runner import save_condition_comparison
save_condition_comparison(all_results, "results/condition_comparison.json")

## 3. Parameter sweep — λ (selfishness)

In [ ]:
lambda_results = {}
for lam in [0.2, 0.5, 0.8, 1.0]:
    cfg = NegotiationConfig(
        condition=Condition.FREE_FORM,
        lambda_selfishness=lam,
        max_rounds=8,
    )
    res = run_batch(5, cfg, base_seed=0, verbose=False)
    s = summarize(res)
    lambda_results[lam] = s
    print(f"λ={lam} | agreement={s['agreement_rate']:.0%} | welfare={s['avg_total_welfare_all']:.1f}")